# 🔮 Prévision des ventes avec statsmodels
## Projet P9 — Librairie Lapage

---

Ce notebook présente une approche de prévision de séries temporelles avec **statsmodels**, une alternative robuste à Prophet.

### Pourquoi statsmodels ?

| Avantage | Description |
|----------|-------------|
| **Robuste** | Pas de dépendance C++ (CmdStan) |
| **Standard** | Bibliothèque Python classique |
| **Flexible** | ARIMA, SARIMAX, Holt-Winters... |
| **Léger** | Installation simple |

### Approches disponibles

| Modèle | Usage |
|--------|-------|
| **Holt-Winters** | Saisonnalité multiplicative/additive |
| **SARIMAX** | ARIMA + saisonnalité |
| **Simple moyenne mobile** | Baseline rapide |

---
## 1. Installation et imports

In [ ]:
# Installation (décommenter si nécessaire)
# !pip install statsmodels plotly scikit-learn

In [1]:
# Imports
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Statsmodels
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Métriques
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("✅ Imports réussis")

✅ Imports réussis


---
## 2. Chargement et préparation des données

In [2]:
# Charger les transactions
df = pd.read_csv('../data/processed/transactions_enrichies.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"📊 {len(df):,} transactions chargées")
print(f"📅 Période : {df['date'].min().date()} → {df['date'].max().date()}")

📊 687,534 transactions chargées
📅 Période : 2021-03-01 → 2023-02-28


In [5]:
# Agrégation par jour : CA journalier
df_daily = df.groupby(pd.Grouper(key='date', freq='D')).agg(
    ca=('price', 'sum'),
    nb_transactions=('price', 'count')
).reset_index()

# Créer un index datetime complet (combler les jours manquants)
date_range = pd.date_range(start=df_daily['date'].min(), end=df_daily['date'].max(), freq='D')
df_daily = df_daily.set_index('date').reindex(date_range).fillna(0)
df_daily.index.name = 'date'
df_daily = df_daily.reset_index()

print(f"📅 {len(df_daily)} jours de données (sans trous)")
df_daily.head()

📅 730 jours de données (sans trous)


,date,ca,nb_transactions
0,2021-03-01,16565.22,962
1,2021-03-02,15486.45,939
2,2021-03-03,15198.69,911
3,2021-03-04,15196.07,903
4,2021-03-05,17471.37,943


In [6]:
# Visualisation du CA journalier
fig = px.line(
    df_daily, x='date', y='ca',
    title='📈 Chiffre d\'affaires journalier — Lapage',
    labels={'date': 'Date', 'ca': 'CA (€)'}
)
fig.update_layout(template='plotly_white')
fig.show()

---
## 3. Analyse de la saisonnalité

Avant de modéliser, analysons les composantes de la série.

In [7]:
# Créer une série temporelle avec index datetime
ts = df_daily.set_index('date')['ca']

# Décomposition saisonnière (période = 7 jours pour hebdomadaire)
decomposition = seasonal_decompose(ts, model='multiplicative', period=7)

print("✅ Décomposition effectuée")

✅ Décomposition effectuée


In [8]:
# Visualisation de la décomposition
fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=['Série originale', 'Tendance', 'Saisonnalité', 'Résidus'],
    vertical_spacing=0.08
)

fig.add_trace(go.Scatter(x=ts.index, y=ts.values, name='Original', line=dict(color='#333')), row=1, col=1)
fig.add_trace(go.Scatter(x=decomposition.trend.index, y=decomposition.trend.values, name='Tendance', line=dict(color='#FF6B6B')), row=2, col=1)
fig.add_trace(go.Scatter(x=decomposition.seasonal.index, y=decomposition.seasonal.values, name='Saisonnalité', line=dict(color='#4ECDC4')), row=3, col=1)
fig.add_trace(go.Scatter(x=decomposition.resid.index, y=decomposition.resid.values, name='Résidus', line=dict(color='#45B7D1')), row=4, col=1)

fig.update_layout(height=800, title_text='📊 Décomposition de la série temporelle', showlegend=False, template='plotly_white')
fig.show()

In [9]:
# Analyse de la saisonnalité hebdomadaire
df_daily['jour_semaine'] = df_daily['date'].dt.dayofweek
ca_par_jour = df_daily.groupby('jour_semaine')['ca'].mean().reset_index()

jours = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']
ca_par_jour['jour'] = ca_par_jour['jour_semaine'].map(lambda x: jours[x])

fig = px.bar(
    ca_par_jour, x='jour', y='ca',
    title='📅 CA moyen par jour de la semaine',
    labels={'jour': 'Jour', 'ca': 'CA moyen (€)'},
    color='ca', color_continuous_scale='Blues'
)
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

---
## 4. Split Train/Test

On garde les 30 derniers jours pour évaluer le modèle.

In [10]:
# Split temporel
horizon = 30  # Jours à prédire

train = df_daily.iloc[:-horizon].copy()
test = df_daily.iloc[-horizon:].copy()

print(f"📊 Train : {len(train)} jours ({train['date'].min().date()} → {train['date'].max().date()})")
print(f"📊 Test  : {len(test)} jours ({test['date'].min().date()} → {test['date'].max().date()})")

📊 Train : 700 jours (2021-03-01 → 2023-01-29)
📊 Test  : 30 jours (2023-01-30 → 2023-02-28)


In [12]:
# Visualisation du split
fig = go.Figure()

fig.add_trace(go.Scatter(x=train['date'], y=train['ca'], name='Train', line=dict(color='#0066FF')))
fig.add_trace(go.Scatter(x=test['date'], y=test['ca'], name='Test', line=dict(color='#FF6B6B')))

#fig.add_vline(x=train['date'].max(), line_dash="dash", line_color="gray", annotation_text="Split")

fig.update_layout(title='📊 Split Train/Test', template='plotly_white', xaxis_title='Date', yaxis_title='CA (€)')
fig.show()

---
## 5. Modèle Holt-Winters (Lissage exponentiel)

Le modèle **Holt-Winters** (ou Triple Exponential Smoothing) capture :
- **Niveau** : valeur de base
- **Tendance** : évolution globale
- **Saisonnalité** : patterns récurrents

In [13]:
# Créer la série d'entraînement
ts_train = train.set_index('date')['ca']

# Modèle Holt-Winters avec saisonnalité hebdomadaire
model = ExponentialSmoothing(
    ts_train,
    trend='add',              # Tendance additive
    seasonal='mul',           # Saisonnalité multiplicative
    seasonal_periods=7,       # Période = 7 jours (hebdomadaire)
    damped_trend=True         # Amortir la tendance (évite extrapolation extrême)
)

print("🔧 Modèle Holt-Winters configuré")

🔧 Modèle Holt-Winters configuré


c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


In [14]:
# Entraînement
model_fit = model.fit(optimized=True)

print("✅ Modèle entraîné !")
print(f"\n📊 Paramètres optimisés :")
print(f"   Alpha (niveau)      : {model_fit.params['smoothing_level']:.4f}")
print(f"   Beta (tendance)     : {model_fit.params['smoothing_trend']:.4f}")
print(f"   Gamma (saisonnalité): {model_fit.params['smoothing_seasonal']:.4f}")
print(f"   Phi (amortissement) : {model_fit.params['damping_trend']:.4f}")

✅ Modèle entraîné !

📊 Paramètres optimisés :
   Alpha (niveau)      : 0.2568
   Beta (tendance)     : 0.0000
   Gamma (saisonnalité): 0.0000
   Phi (amortissement) : 0.8512


---
## 6. Prévisions

In [15]:
# Prévisions sur l'horizon de test
forecast = model_fit.forecast(horizon)

print(f"📅 Prévisions générées : {len(forecast)} jours")
forecast.head(10)

📅 Prévisions générées : 30 jours


2023-01-30    16898.979615
2023-01-31    16764.810826
2023-02-01    16766.488893
2023-02-02    16772.268303
2023-02-03    16721.237606
2023-02-04    16713.889540
2023-02-05    16802.321834
2023-02-06    16898.979615
2023-02-07    16764.810826
2023-02-08    16766.488893
Freq: D, dtype: float64

In [16]:
# Créer un DataFrame de prévisions
df_forecast = pd.DataFrame({
    'date': test['date'].values,
    'ca_actual': test['ca'].values,
    'ca_predicted': forecast.values
})

df_forecast.head(10)

,date,ca_actual,ca_predicted
0,2023-01-30,16638.34,16898.979615
1,2023-01-31,16338.90,16764.810826
2,2023-02-01,16718.43,16766.488893
3,2023-02-02,17423.19,16772.268303
4,2023-02-03,15669.55,16721.237606
5,2023-02-04,14586.31,16713.889540
6,2023-02-05,16111.41,16802.321834
7,2023-02-06,14235.62,16898.979615
8,2023-02-07,18217.02,16764.810826
9,2023-02-08,15397.83,16766.488893


In [17]:
# Visualisation : prévisions vs réel
fig = go.Figure()

# Données d'entraînement (derniers 60 jours)
train_recent = train.tail(60)
fig.add_trace(go.Scatter(
    x=train_recent['date'], y=train_recent['ca'],
    name='Historique', line=dict(color='#333')
))

# Données de test (réelles)
fig.add_trace(go.Scatter(
    x=df_forecast['date'], y=df_forecast['ca_actual'],
    name='Réel', line=dict(color='#FF6B6B', width=2)
))

# Prévisions
fig.add_trace(go.Scatter(
    x=df_forecast['date'], y=df_forecast['ca_predicted'],
    name='Prévision', line=dict(color='#0066FF', width=2, dash='dash')
))

# Ligne de séparation
fig.add_vline(x=train['date'].max(), line_dash="dash", line_color="gray")

fig.update_layout(
    title='🔮 Prévisions vs Réel — Holt-Winters',
    xaxis_title='Date',
    yaxis_title='CA (€)',
    template='plotly_white',
    hovermode='x unified'
)
fig.show()

---
## 7. Évaluation du modèle

In [18]:
# Métriques d'évaluation
y_true = df_forecast['ca_actual']
y_pred = df_forecast['ca_predicted']

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true.replace(0, np.nan))) * 100

print("="*50)
print("📏 MÉTRIQUES D'ÉVALUATION (sur les 30 derniers jours)")
print("="*50)
print(f"MAE  (Mean Absolute Error)      : {mae:,.2f} €")
print(f"RMSE (Root Mean Square Error)   : {rmse:,.2f} €")
print(f"MAPE (Mean Abs Percentage Error): {mape:.2f} %")
print("="*50)
print(f"\n💡 En moyenne, l'erreur de prévision est de {mae:,.0f}€ par jour")
print(f"💡 L'erreur relative moyenne est de {mape:.1f}%")

📏 MÉTRIQUES D'ÉVALUATION (sur les 30 derniers jours)
MAE  (Mean Absolute Error)      : 1,129.08 €
RMSE (Root Mean Square Error)   : 1,332.67 €
MAPE (Mean Abs Percentage Error): 7.02 %

💡 En moyenne, l'erreur de prévision est de 1,129€ par jour
💡 L'erreur relative moyenne est de 7.0%


In [19]:
# Visualisation des erreurs
df_forecast['erreur'] = df_forecast['ca_actual'] - df_forecast['ca_predicted']
df_forecast['erreur_pct'] = (df_forecast['erreur'] / df_forecast['ca_actual']) * 100

fig = make_subplots(rows=2, cols=1, subplot_titles=['Erreur absolue (€)', 'Erreur relative (%)'])

fig.add_trace(go.Bar(x=df_forecast['date'], y=df_forecast['erreur'], name='Erreur €', marker_color='#FF6B6B'), row=1, col=1)
fig.add_trace(go.Bar(x=df_forecast['date'], y=df_forecast['erreur_pct'], name='Erreur %', marker_color='#4ECDC4'), row=2, col=1)

fig.update_layout(height=500, title_text='📊 Distribution des erreurs', showlegend=False, template='plotly_white')
fig.show()

---
## 8. Intervalles de confiance

On peut estimer les intervalles de confiance à partir des résidus.

In [20]:
# Calculer l'écart-type des résidus sur l'entraînement
residuals = model_fit.resid
std_residuals = residuals.std()

print(f"📊 Écart-type des résidus : {std_residuals:,.2f} €")

# Intervalles de confiance (95%)
z = 1.96  # 95% confidence
df_forecast['lower'] = df_forecast['ca_predicted'] - z * std_residuals
df_forecast['upper'] = df_forecast['ca_predicted'] + z * std_residuals

# Ne pas avoir de valeurs négatives
df_forecast['lower'] = df_forecast['lower'].clip(lower=0)

df_forecast[['date', 'ca_predicted', 'lower', 'upper']].head(10)

📊 Écart-type des résidus : 1,213.55 €


,date,ca_predicted,lower,upper
0,2023-01-30,16898.979615,14520.425443,19277.533787
1,2023-01-31,16764.810826,14386.256654,19143.364998
2,2023-02-01,16766.488893,14387.934722,19145.043065
3,2023-02-02,16772.268303,14393.714131,19150.822475
4,2023-02-03,16721.237606,14342.683434,19099.791778
5,2023-02-04,16713.889540,14335.335368,19092.443712
6,2023-02-05,16802.321834,14423.767662,19180.876005
7,2023-02-06,16898.979615,14520.425443,19277.533787
8,2023-02-07,16764.810826,14386.256654,19143.364998
9,2023-02-08,16766.488893,14387.934722,19145.043065


In [21]:
# Visualisation avec intervalles de confiance
fig = go.Figure()

# Intervalle de confiance
fig.add_trace(go.Scatter(
    x=pd.concat([df_forecast['date'], df_forecast['date'][::-1]]),
    y=pd.concat([df_forecast['upper'], df_forecast['lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(0, 100, 255, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='IC 95%'
))

# Prévisions
fig.add_trace(go.Scatter(
    x=df_forecast['date'], y=df_forecast['ca_predicted'],
    name='Prévision', line=dict(color='#0066FF', width=2)
))

# Réel
fig.add_trace(go.Scatter(
    x=df_forecast['date'], y=df_forecast['ca_actual'],
    name='Réel', mode='markers', marker=dict(color='#FF6B6B', size=8)
))

fig.update_layout(
    title='🔮 Prévisions avec intervalle de confiance 95%',
    xaxis_title='Date',
    yaxis_title='CA (€)',
    template='plotly_white'
)
fig.show()

---
## 9. Réentraînement sur toutes les données + prévisions futures

In [22]:
# Réentraîner sur TOUTES les données
ts_full = df_daily.set_index('date')['ca']

model_full = ExponentialSmoothing(
    ts_full,
    trend='add',
    seasonal='mul',
    seasonal_periods=7,
    damped_trend=True
)

model_full_fit = model_full.fit(optimized=True)
print("✅ Modèle réentraîné sur toutes les données")

✅ Modèle réentraîné sur toutes les données


c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


In [23]:
# Prévisions pour les 30 prochains jours
horizon_future = 30
forecast_future = model_full_fit.forecast(horizon_future)

# Créer les dates futures
last_date = df_daily['date'].max()
future_dates = pd.date_range(start=last_date + timedelta(days=1), periods=horizon_future, freq='D')

# DataFrame des prévisions futures
df_future = pd.DataFrame({
    'date': future_dates,
    'ca_predicted': forecast_future.values,
    'lower': forecast_future.values - z * std_residuals,
    'upper': forecast_future.values + z * std_residuals
})
df_future['lower'] = df_future['lower'].clip(lower=0)

print(f"📅 Prévisions du {df_future['date'].min().date()} au {df_future['date'].max().date()}")
df_future

📅 Prévisions du 2023-03-01 au 2023-03-30


,date,ca_predicted,lower,upper
0,2023-03-01,17077.154580,14698.600409,19455.708752
1,2023-03-02,17091.767385,14713.213214,19470.321557
2,2023-03-03,17006.682527,14628.128355,19385.236699
3,2023-03-04,17047.768589,14669.214417,19426.322761
4,2023-03-05,17134.009093,14755.454921,19512.563264
5,2023-03-06,17209.369926,14830.815754,19587.924098
6,2023-03-07,17139.789433,14761.235261,19518.343605
7,2023-03-08,17077.154580,14698.600409,19455.708752
8,2023-03-09,17091.767385,14713.213214,19470.321557
9,2023-03-10,17006.682527,14628.128355,19385.236699


In [25]:
# Visualisation complète
fig = go.Figure()

# Historique (derniers 90 jours)
recent = df_daily.tail(90)
fig.add_trace(go.Scatter(
    x=recent['date'], y=recent['ca'],
    name='Historique', line=dict(color='#333')
))

# Intervalle de confiance
fig.add_trace(go.Scatter(
    x=pd.concat([df_future['date'], df_future['date'][::-1]]),
    y=pd.concat([df_future['upper'], df_future['lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(0, 100, 255, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='IC 95%'
))

# Prévisions
fig.add_trace(go.Scatter(
    x=df_future['date'], y=df_future['ca_predicted'],
    name='Prévision', line=dict(color='#0066FF', width=2)
))

# Ligne de séparation
#fig.add_vline(x=last_date, line_dash="dash", line_color="red", annotation_text="Début prévisions")

fig.update_layout(
    title='🔮 Prévisions des 30 prochains jours — Holt-Winters',
    xaxis_title='Date',
    yaxis_title='CA (€)',
    template='plotly_white',
    hovermode='x unified'
)
fig.show()

---
## 10. Sauvegarde du modèle

In [ ]:
import pickle
from pathlib import Path

# Créer le dossier models si nécessaire
models_dir = Path('../models/saved')
models_dir.mkdir(parents=True, exist_ok=True)

# Sauvegarder le modèle + métadonnées
model_data = {
    'model': model_full_fit,
    'std_residuals': std_residuals,
    'last_date': last_date,
    'metrics': {
        'mae': mae,
        'rmse': rmse,
        'mape': mape
    },
    'params': {
        'seasonal_periods': 7,
        'trend': 'add',
        'seasonal': 'mul'
    }
}

model_path = models_dir / 'holt_winters_model.pkl'

with open(model_path, 'wb') as f:
    pickle.dump(model_data, f)

print(f"✅ Modèle sauvegardé : {model_path}")
print(f"   Taille : {model_path.stat().st_size / 1024:.1f} Ko")

In [ ]:
# Vérification : recharger et tester
with open(model_path, 'rb') as f:
    loaded = pickle.load(f)

# Test de prévision
test_forecast = loaded['model'].forecast(7)

print("✅ Modèle rechargé avec succès !")
print(f"\n📊 Dernière date d'entraînement : {loaded['last_date'].date()}")
print(f"📏 Métriques : MAE={loaded['metrics']['mae']:.2f}€, MAPE={loaded['metrics']['mape']:.2f}%")
print(f"\n📅 Prévisions des 7 prochains jours :")
test_forecast

---
## 11. Fonction utilitaire pour l'API

In [ ]:
def predict_ca(model_data: dict, horizon_days: int = 30) -> dict:
    """
    Génère des prévisions de CA pour les N prochains jours.
    
    Parameters
    ----------
    model_data : dict
        Dictionnaire contenant le modèle et ses métadonnées
    horizon_days : int
        Nombre de jours à prédire
        
    Returns
    -------
    dict : Prévisions au format JSON-ready
    """
    model = model_data['model']
    std = model_data['std_residuals']
    last_date = model_data['last_date']
    
    # Générer les prévisions
    forecast = model.forecast(horizon_days)
    
    # Dates futures
    future_dates = pd.date_range(
        start=last_date + timedelta(days=1),
        periods=horizon_days,
        freq='D'
    )
    
    # Formater pour l'API
    predictions = []
    for date, yhat in zip(future_dates, forecast):
        predictions.append({
            "date": date.strftime('%Y-%m-%d'),
            "ca_predicted": round(float(yhat), 2),
            "lower": round(max(0, float(yhat - 1.96 * std)), 2),
            "upper": round(float(yhat + 1.96 * std), 2)
        })
    
    return {
        "horizon_days": horizon_days,
        "predictions": predictions,
        "model_info": {
            "type": "Holt-Winters",
            "last_training_date": last_date.strftime('%Y-%m-%d'),
            "metrics": model_data['metrics']
        }
    }

# Test
result = predict_ca(loaded, horizon_days=7)
print("📦 Format de sortie API :")
result

---
## 📝 Résumé

### Ce que nous avons fait

| Étape | Description |
|-------|-------------|
| **Préparation** | CA journalier, index datetime complet |
| **Analyse** | Décomposition saisonnière |
| **Modèle** | Holt-Winters (ExponentialSmoothing) |
| **Évaluation** | MAE, RMSE, MAPE sur 30 jours |
| **Prévision** | 30 jours avec IC 95% |
| **Sauvegarde** | `holt_winters_model.pkl` |

### Avantages de cette approche

| ✅ | Description |
|---|-------------|
| **Robuste** | Pas de dépendance CmdStan |
| **Rapide** | Entraînement en quelques secondes |
| **Interprétable** | Paramètres alpha, beta, gamma explicites |
| **Léger** | Modèle < 10 Ko |

### Prochaines étapes

1. **Intégrer à l'API** : Endpoint `POST /api/predict`
2. **Consommer depuis le dashboard** : Appel API + visualisation Plotly

---

✅ **Notebook terminé** — Prêt pour l'intégration API !